# Cleaning details.csv

In [4]:
import pandas as pd
import os
from unidecode import unidecode
import ast

In [5]:
# path
RAW_PATH = os.path.join("../..", "dataset", "raw", "details.csv")
CLEAN_PATH = os.path.join("../..", "dataset", "cleaned", "details_cleaned.csv")

In [6]:
df = pd.read_csv(RAW_PATH)
original_rows = len(df)

# id
df = df.dropna(subset=['mal_id'])
df['mal_id'] = pd.to_numeric(df['mal_id'], errors='coerce').astype(int)

# rendiamo esplicita l'informazione mancante
df['title'] = df['title'].fillna("Unknown")
df['type'] = df['type'].fillna("Unknown")
df['status'] = df['status'].fillna("Unknown")

def normalize_text(x):
    if isinstance(x, str):
        return unidecode(x).strip()
    return x

df['title'] = df['title'].apply(normalize_text)
df['type'] = df['type'].apply(normalize_text)
df['status'] = df['status'].apply(normalize_text)

# conversione tipi
df['type'] = df['type'].astype("string")
df['status'] = df['status'].astype("string")

df['score'] = pd.to_numeric(df['score'], errors='coerce')
df['scored_by'] = pd.to_numeric(df['scored_by'], errors='coerce')

df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')

# pulizia genres
# valori mancati -> lista vuota
df['genres'] = df['genres'].fillna("[]")

def parse_genres(x):
    # se è gia una lista, la restituiamo
    if isinstance(x, list):
        return x
    # convertiamo la stringa in lista
    try:
        return ast.literal_eval(x)
    except:
        return []
# parsing stringa -> lista
df['genres'] = df['genres'].apply(parse_genres)
# se non ci sono generi, rendiamo esplicita l'incertezza
df['genres'] = df['genres'].apply(
    lambda g: ['Unknown'] if len(g) == 0 else g
)
# normalizzazione dei nomi dei generi
df['genres'] = df['genres'].apply(
    lambda g: [genre.strip().title() for genre in g]
)

#df = df.drop_duplicates()
df = df.drop_duplicates(subset=['mal_id'])

print("Righe iniziali:", original_rows)
print("Righe finali:", len(df))
print("Righe rimosse:", original_rows - len(df))

df.to_csv(CLEAN_PATH, index=False, encoding="utf-8")


Righe iniziali: 28955
Righe finali: 28955
Righe rimosse: 0
